<a href="https://colab.research.google.com/github/GreJangre/Baekjoon_Swift/blob/master/Colab_%EC%8B%9C%EC%9E%91%ED%95%98%EA%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import sys
import time
from datetime import datetime, timedelta
from typing import Dict, List

import configparser
import requests

from urllib import parse


# 경로
PATH_CONFIG = '.'
PATH_DATA = '.'


BASE_URL = "https://opendart.fss.or.kr/api"
LIST_URL = f"{BASE_URL}/list.json"        # 공시검색
COMPANY_URL = f"{BASE_URL}/company.json"  # 기업개황

# 조회할 법인구분 (Y:유가증권, K:코스닥)
TARGET_CORP_CLS = ["Y", "K"]

# .dat 파일 저장 설정
OUTPUT_DIR = "."          # 저장 폴더 (Colab 기본 작업 폴더: /content)
DELIMITER = "|"           # 구분자
DOWNLOAD_IN_COLAB = True  # Colab에서 실행 시 생성된 파일을 브라우저로 다운로드

PAGE_COUNT = 100        # 공시검색 페이지당 최대 건수
REQUEST_INTERVAL = 0.2  # 호출 간 대기(초) - 호출 제한 대비
TIMEOUT = 15

# OpenDART 응답 status 코드 (주요 항목)
STATUS_OK = "000"
STATUS_NO_DATA = "013"
STATUS_MESSAGES = {
    "010": "등록되지 않은 인증키입니다.",
    "011": "사용할 수 없는 인증키입니다.",
    "012": "접근할 수 없는 IP입니다.",
    "013": "조회된 데이터가 없습니다.",
    "014": "파일이 존재하지 않습니다.",
    "020": "요청 제한을 초과하였습니다.",
    "021": "조회 가능한 회사 개수가 초과하였습니다.",
    "100": "필드의 부적절한 값입니다.",
    "101": "부적절한 접근입니다.",
    "800": "시스템 점검으로 인한 서비스가 중지 중입니다.",
    "900": "정의되지 않은 오류가 발생하였습니다.",
    "901": "개인정보 보유기간이 만료되어 사용할 수 없는 인증키입니다.",
}

# API 인증키
config = configparser.ConfigParser()
config.read(PATH_CONFIG + 'config.ini')
API_KEY = parse.quote(config['OPENDART']['key'])


class DartApiError(Exception):
    pass


def write_dat(path: str, rows: List[dict]) -> None:
    """구분자 기반 .dat 파일 저장 (UTF-8, 헤더 없음). API 응답 필드를 모두 저장."""
    # 컬럼은 응답에 나타난 순서대로 결정 (행마다 누락된 필드는 빈 값)
    fields = []  # type: List[str]
    for row in rows:
        for k in row:
            if k not in fields:
                fields.append(k)

    def clean(v) -> str:
        # 값 안의 구분자/줄바꿈은 공백으로 치환해 컬럼이 깨지지 않게 함
        text = "" if v is None else str(v)
        return text.replace(DELIMITER, " ").replace("\r", " ").replace("\n", " ").strip()

    with open(path, "w", encoding="utf-8", newline="") as f:
        for row in rows:
            f.write(DELIMITER.join(clean(row.get(k)) for k in fields) + "\n")


def get_yesterday_kst() -> str:
    """한국 시간 기준 전일자(YYYYMMDD)."""
    now_kst = datetime.utcnow() + timedelta(hours=9)  # UTC+9 (zoneinfo 미사용)
    return (now_kst - timedelta(days=1)).strftime("%Y%m%d")


def call_api(url: str, params: dict) -> dict:
    """OpenDART 호출 공통 함수. status가 000/013 이외이면 예외 발생."""
    resp = requests.get(url, params=params, timeout=TIMEOUT)
    resp.raise_for_status()
    data = resp.json()

    status = data.get("status")
    if status not in (STATUS_OK, STATUS_NO_DATA):
        msg = STATUS_MESSAGES.get(status, data.get("message", ""))
        raise DartApiError(f"[{status}] {msg}")
    return data


def fetch_disclosures(api_key: str, target_date: str, corp_cls: str) -> List[dict]:
    """특정 일자 + 법인구분의 공시 목록 전체 조회 (페이지네이션 처리)."""
    results = []  # type: List[dict]
    page_no = 1

    while True:
        params = {
            "crtfc_key": api_key,
            "bgn_de": target_date,
            "end_de": target_date,
            "corp_cls": corp_cls,
            "page_no": page_no,
            "page_count": PAGE_COUNT,
        }
        data = call_api(LIST_URL, params)

        if data["status"] == STATUS_NO_DATA:
            break

        results.extend(data.get("list", []))

        total_page = int(data.get("total_page", 1))
        if page_no >= total_page:
            break
        page_no += 1
        time.sleep(REQUEST_INTERVAL)

    return results


def fetch_company_info(api_key: str, corp_code: str) -> dict:
    """고유번호(corp_code, 8자리) 기준 기업개황 조회."""
    params = {"crtfc_key": api_key, "corp_code": corp_code}
    return call_api(COMPANY_URL, params)


def main() -> int:
    target_date = get_yesterday_kst()
    print(f"[1] {target_date} 공시 목록 조회 중 (법인구분: {', '.join(TARGET_CORP_CLS)})...")

    disclosures = []
    try:
        for corp_cls in TARGET_CORP_CLS:
            items = fetch_disclosures(API_KEY, target_date, corp_cls)
            print(f"    -> [{corp_cls}] {len(items)}건")
            disclosures.extend(items)
            time.sleep(REQUEST_INTERVAL)
    except (DartApiError, requests.RequestException) as e:
        print(f"공시 조회 실패: {e}", file=sys.stderr)
        return 1

    # 접수번호 기준 정렬 (접수 순서)
    disclosures.sort(key=lambda d: d.get("rcept_no", ""))
    print(f"    -> 합계 {len(disclosures)}건 ({'/'.join(TARGET_CORP_CLS)})")
    if not disclosures:
        print("    (주말/공휴일이거나 공시가 없는 날입니다.)")
        return 0

    # 같은 기업이 여러 건 공시할 수 있으므로 corp_code 기준으로 중복 제거
    corp_codes = sorted({d["corp_code"] for d in disclosures})
    print(f"[2] 고유번호 {len(corp_codes)}개 기업개황 조회 중...")

    company_map = {}  # type: Dict[str, dict]
    for i, corp_code in enumerate(corp_codes, start=1):
        try:
            company_map[corp_code] = fetch_company_info(API_KEY, corp_code)
        except (DartApiError, requests.RequestException) as e:
            print(f"    ! {corp_code} 기업개황 조회 실패: {e}", file=sys.stderr)
            company_map[corp_code] = {}
            # 호출 제한 초과 시 더 진행해도 의미가 없으므로 중단
            if isinstance(e, DartApiError) and str(e).startswith("[020]"):
                print("    ! 요청 제한 초과로 조회를 중단합니다.", file=sys.stderr)
                break
        if i % 50 == 0:
            print(f"    ... {i}/{len(corp_codes)}")
        time.sleep(REQUEST_INTERVAL)

    # .dat 파일 저장 (조회 내용별로 파일 분리)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    disclosure_path = os.path.join(OUTPUT_DIR, "disclosure_{}.dat".format(target_date))
    company_path = os.path.join(OUTPUT_DIR, "company_{}.dat".format(target_date))

    company_rows = []
    for code, c in sorted(company_map.items()):
        if not c:
            continue
        row = {"corp_code": code}  # 응답에 없는 고유번호를 맨 앞에 추가
        row.update((k, v) for k, v in c.items() if k not in ("status", "message"))
        company_rows.append(row)
    write_dat(disclosure_path, disclosures)
    write_dat(company_path, company_rows)
    print("[3] 파일 저장 완료")
    print("    - {} ({}건)".format(disclosure_path, len(disclosures)))
    print("    - {} ({}건)".format(company_path, len(company_rows)))

    if DOWNLOAD_IN_COLAB and "google.colab" in sys.modules:
        from google.colab import files
        files.download(disclosure_path)
        files.download(company_path)

    return 0


if __name__ == "__main__":
        main()

조회 기간: 2026-07-29 ~ 2026-08-27 (요일 무관, 총 30일)
저장 폴더: krx_raw_json_30days/

[20260729] 유가증권   -> OK, 943건, 저장: krx_raw_json_30days/kospi_bydd_trd_20260729.json
[20260729] 코스닥    -> OK, 1821건, 저장: krx_raw_json_30days/kosdaq_bydd_trd_20260729.json
[20260729] 코넥스    -> OK, 109건, 저장: krx_raw_json_30days/konex_bydd_trd_20260729.json
[20260801] 유가증권   -> OK, 0건, 저장: krx_raw_json_30days/kospi_bydd_trd_20260801.json
[20260801] 코스닥    -> OK, 0건, 저장: krx_raw_json_30days/kosdaq_bydd_trd_20260801.json
[20260801] 코넥스    -> OK, 0건, 저장: krx_raw_json_30days/konex_bydd_trd_20260801.json
[20260804] 유가증권   -> OK, 943건, 저장: krx_raw_json_30days/kospi_bydd_trd_20260804.json
[20260804] 코스닥    -> OK, 1820건, 저장: krx_raw_json_30days/kosdaq_bydd_trd_20260804.json
[20260804] 코넥스    -> OK, 109건, 저장: krx_raw_json_30days/konex_bydd_trd_20260804.json
[20260807] 유가증권   -> OK, 943건, 저장: krx_raw_json_30days/kospi_bydd_trd_20260807.json
[20260807] 코스닥    -> OK, 1820건, 저장: krx_raw_json_30days/kosdaq_bydd_trd_20260807.json
